# The SVM vs. Logistic Regression Showdown

In this lab, you will practice working with non-linear kernels combined with logistic regression and SVM classifiers. The goal is to compare these commonly used techniques. Which comes out on top in terms of accuracy? Runtime? Is there much of a difference at all?  

## Loading the Data

First, we load all the packages we'll need.

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics.pairwise import pairwise_kernels
import scipy
from sklearn import svm, linear_model
from sklearn.model_selection import GridSearchCV
import time

Again we download the data from the Tensorflow package, which you will need to install.  You can get the data from other sources as well.

In the Tensorflow dataset, the training and test data are represented as arrays:

     Xtr.shape = 60000 x 28 x 28
     Xts.shape = 10000 x 28 x 28
     
The test data consists of `60000` images of size `28 x 28` pixels; the test data consists of `10000` images.

In [2]:
import tensorflow as tf

(Xtr_raw,ytr),(Xts_raw,yts) = tf.keras.datasets.mnist.load_data()

print('Xtr shape: %s' % str(Xtr_raw.shape))
print('Xts shape: %s' % str(Xts_raw.shape))

ntr = Xtr_raw.shape[0]
nts = Xts_raw.shape[0]
nrow = Xtr_raw.shape[1]
ncol = Xtr_raw.shape[2]

Xtr shape: (60000, 28, 28)
Xts shape: (10000, 28, 28)


Each pixel value is from `[0,255]`.  For this lab, we recale the values to lie between -1 to 1 and reshape  the data to `ntr x npix` and `nts x npix`.

In [3]:
npix = nrow*ncol
Xtr = Xtr_raw.reshape((ntr,npix))
print(Xtr[1,:])
Xtr = (Xtr/255 - .5)
print(Xtr[1,:])

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0  51 159 253 159  50   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0  48 238 252 252 252 237   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0  54 227 253 252 239 233 252  57   6   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0  10  60 224 252 253 252 202  84 252
 253 122   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0 163 252 252 252 253 252 252  96 189 253 167   

In [4]:
npix = nrow*ncol
Xtr = (Xtr_raw/255 - 0.5)
Xtr = Xtr.reshape((ntr,npix))

Xts = (Xts_raw/255 - 0.5)
Xts = Xts.reshape((nts,npix))

In [5]:
print(Xtr)

[[-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]
 [-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]
 [-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]
 ...
 [-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]
 [-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]
 [-0.5 -0.5 -0.5 ... -0.5 -0.5 -0.5]]


For this lab we're only going to use a fraction of the MNIST data -- otherwise our models will take too much time and memory to run. Using only part of the training data will of course lead to worse results. Given enough computational resources and time, we would ideally be running on the full data set. The follow code creates a new test and train set, with 10000 examples for train and 5000 for test.

In [6]:
ntr1 = 10000
nts1 = 5000
Iperm = np.random.permutation(ntr1)
Xtr1 = Xtr[Iperm[:ntr1],:]
ytr1 = ytr[Iperm[:ntr1]]
Iperm = np.random.permutation(nts1)
Xts1 = Xts[Iperm[:nts1],:]
yts1 = yts[Iperm[:nts1]]

## Problem set up and establishing a baseline

To simplify the problem (and speed things up) we're also going to restrict to binary classification. In particular, let's try to design classifier a that separates the 8's from all other digits.

Create binary 0/1 label vectors `ytr8` and `yts8` which are 1 wherever `ytr1` and `yts1` equal 8, and 0 everywhere else.

In [7]:
# TODO

ytr8 = np.where(ytr1 == 8, 1, 0)
yts8 = np.where(yts1 == 8, 1, 0)
# print(ytr8, yts8)

# ytr8 =
# yts8 =

Most of the digits in the test dataset aren't equal to 8. So if we simply guess 0 for every image in `Xts`, we might expect to get classification accuracy around 90%. Our goal should be to significantly beat this **baseline**.

Formally, write a few lines of code to check what test error would be achieved by the all zeros classifier.

In [8]:
# TODO
acc = np.mean(yts8 == 0)
print('Accuaracy = {0:f}'.format(acc))
# ...
# acc =
# print('Accuaracy = {0:f}'.format(acc))

Accuaracy = 0.902200


As a second baseline, let's see how we do with standard (non-kernel) logistic regression. As in the MNIST demo, you can use `scikit-learn`'s built in function `linear_model.LogisticRegression` to fit the model and compute the accuracy. Use no regularization and the `lbfgs` solver. You should acheive an improvement to around 93-95%.

In [9]:
# TODO
# ...
acc = linear_model.LogisticRegression(penalty = 'l2', solver = "lbfgs").fit(Xtr1, ytr8).score(Xts1, yts8)
# acc =
print('Logistic Regression Accuaracy = {0:f}'.format(acc))

Logistic Regression Accuaracy = 0.944400


C:\Users\Dove0u0\.conda\envs\mariopy\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Kernel Logistic Regression

To improve on this baseline performance, let's try using the logistic regression classifier with a *non-linear* kernel. Recall from class that any non-linear kernel similarity function $k(\vec{w},\vec{z})$ is equal to $\phi(\vec{w})^T\phi(\vec{z})$ for some feature transformation $\phi$. However, we typically do not need to compute this feature tranformation explicitly: instead we can work directly with the kernel gram matrix $K \in \mathbb{R}^{n\times n}$. Recall that $K_{i,j} = k(\vec{x}_i,\vec{x}_j)$ where $\vec{x}_i$ is the $i^\text{th}$ training data point.

For this lab we will be using the radial basis function kernel. For a given scaling factor $\gamma$ this kernel is defined as:
$$
k(\vec{w},\vec{z}) = e^{-\gamma\|\vec{w}-\vec{z}\|_2^2}
$$

In [10]:
def rbf_kernel(w,z,gamma):
    d = w - z
    return np.exp(-gamma*np.sum(d*d))

Construct the kernel matrix `K1` for `Xtr1` with `gamma = .05`.

In [11]:
# TODO
gamma = .05
K1 = np.zeros((ntr1, ntr1))

for i in range(ntr1):
    for j in range(ntr1):
        K1[i,j] = rbf_kernel(Xtr1[i], Xtr1[j], gamma)
# K1 =

If you used a for loop (which is fine) your code might take several minutes to run! Part of the issue is that Python won't know to properly parallize your for loop. For this reason, when constructing kernel matrices it is often faster to us a built-in, carefully optimized function with explicit parallelization. Scikit learn provides such a function through their `metrics` library.

Referring to the documentation here
https://scikit-learn.org/stable/modules/metrics.html#metrics, use this built in function to recreate the same kernel matrix you did above. Store the result at `K`.

In [12]:
# TODO
# K =
K = pairwise_kernels(Xtr1, metric='rbf', gamma=gamma)

Check that you used the function correctly by writing code to confirm that `K = K1`, or at least that the two are equal up to very small differences (which could arise due to numerical precision issues). Try to do this **without a for loop** so that the equality check takes advantage of paralellism.

In [13]:
# TODO
if np.allclose(K, K1, atol=1e-8):
    print("K and K1 are equal")
else:
    print("K and K1 are not equal")

K and K1 are equal


When using a non-linear kernel, it is important to check that you have chosen reasonable parameters (in our case the only parameter is `gamma`). We typically do not want $k(\vec{x}_i,\vec{x}_j)$ to be either negligably small, or very large for all $i\neq j$ in our data set, or we won't be able to learn anything. For the RBF kernel this means that, for any $\vec{x}_i$ we don't want $k(\vec{x}_i,\vec{x}_j)$ very close to 1 (e.g. .9999) for all $j$, or very close to $0$ (e.g. 1e-8) for all $j$.

Let's just check that we're in good shape for the first data vector $\vec{x}_0$. Do so by printing out the 10 largest and 10 smallest values of $k(\vec{x}_0,\vec{x}_j)$ for $j\neq 0$. Note that we always have $k(\vec{x}_0,\vec{x}_0) = 1$ for the RBF kernel.

In [14]:
# TODO
max_10k = [idx for idx in np.argsort(K1[0])[-10:] if idx != 0]
min_10k = [idx for idx in np.argsort(K1[0])[:10] if idx != 0]

print("10 largest similarities (i=0, j!=0):")
for idx in max_10k:
    print(f"Index: {idx}, Value: {K1[0, idx]}")

print("\n10 smallest similarities (i=0, j!=0):")
for idx in min_10k:
    print(f"Index: {idx}, Value: {K1[0, idx]}")

10 largest similarities (i=0, j!=0):
Index: 7595, Value: 0.130277307554015
Index: 8492, Value: 0.13904183885901075
Index: 9048, Value: 0.14451086049080747
Index: 3447, Value: 0.16492955560238384
Index: 4890, Value: 0.1803792717807198
Index: 764, Value: 0.1991740294055558
Index: 9767, Value: 0.20743019618279632
Index: 8977, Value: 0.21833835275524965
Index: 1424, Value: 0.27111884799036123

10 smallest similarities (i=0, j!=0):
Index: 2581, Value: 0.00010907911213879196
Index: 6437, Value: 0.00013166229554677247
Index: 7634, Value: 0.0001318898764892043
Index: 5402, Value: 0.00013849282461366976
Index: 1438, Value: 0.00014051859859977745
Index: 1632, Value: 0.00014257926650298127
Index: 4037, Value: 0.000148731035939322
Index: 4054, Value: 0.00017259179555666583
Index: 3886, Value: 0.00017485222027144063
Index: 2893, Value: 0.00019807708461173253


### Implementation
Maybe surprisingly Scikit learn does not have an implementation for kernel logistic regression. So we have to implement our own!

Write a function function `log_fit` that minimizes the $\ell_2$-regularized logisitic regression loss:
$$
L(\boldsymbol{\alpha}) =\sum_{i=1}^n (1-y_i)(\phi(\mathbf{x}_i)^T\phi(\mathbf{X})^T\vec{\alpha}) - \log(h(\phi(\mathbf{x}_i)^T\phi(\mathbf{X})^T\boldsymbol{\alpha})) + \lambda \|\phi(\mathbf{X})^T\boldsymbol{\alpha}\|_2^2.
$$
As input it takes an $n\times n$ kernel matrix $K$ for the training data, an $n$ length vector `y` of binary class labels, and regularization parameter `lamb`.

To implement your function you can either use your own implementation of gradient descent or used a built in minimizer from `scipy.optimize.minimize`. I recommend trying the later approach. You could try using e.g. the same L-BGFG method we used earlier. In either case, you will need to write a function to compute the gradient of the logistic regression loss above. Most of the methods in `scipy.optimize.minimize` require a function that computes the gradient as input.

In [15]:
# TODO
# def log_fit(K,y,lamb):
#    Function which minizes the regularized logistic regression loss for a given kernel matrix K and target vector y
#    Return the optimal parameters alpha of the logisitic regression model.

def log_fit(K, y, lamb):
  n = len(y)
  def sigmoid(z):
    return 1 / (1 + np.exp(-z))

  def loss(alpha):
    linear_term = np.dot(K, alpha)
    h = sigmoid(linear_term)
    term1 = np.sum((1 - y) * linear_term)
    term2 = -np.sum(np.log(h))
    reg_term = lamb * np.dot(alpha, np.dot(K, alpha))
    return term1 + term2 + reg_term

  def gradient(alpha):
    linear_term = np.dot(K, alpha)
    h = sigmoid(linear_term)
    grad = -np.dot(K, (y - h)) + 2 * lamb * np.dot(K, alpha)
    return grad
  alpha = np.zeros(n)
  result = scipy.optimize.minimize(loss, alpha, method='L-BFGS-B', jac=gradient).x

  return result



Use the `log_fit` function defined above to find parameters `alpha` for the kernel logistic regression model using `lamb = 0` and `K` as constructed above (with `gamma = .05`).

In [16]:
# TODO
# alpha =

alpha = log_fit(K, ytr8, lamb=0)

Suppose we have a test dataset with $m$ examples $\vec{w}_1,\ldots, \vec{w}_m$. Once we obtain a coefficient vector $\alpha$, making predictions for any $\vec{w}_j$ in the test set requires computing:
$$
{y}_{j} = \sum_{i=1}^n \alpha_i \cdot k(\vec{w}_{j}, \vec{x}_i).
$$
where $\vec{x}_1, \ldots \vec{x}_n$ are our training data vectors. We classify $\vec{w}_{j}$ in class 0 if ${y}_{j} \leq 0$ and in class 1 if ${y}_{j} > 0$.

This computation can be rewritten in matrix form as follows:
$$
\vec{y}_{test} = K_{test}\vec{\alpha},
$$
where $\vec{y}_{text}$ is an $m$ length vector and $K_{test}$ is a $m\times n$ matrix whose $(j,i)$ entry is equal to $k(\vec{w}_{j}, \vec{x}_i)$. We classify $\vec{w}_{j}$ in class 0 if $\vec{y}_{test}[j] \leq 0$ and in class 1 if $\vec{y}_{test}[j] > 0$.


Use the `pairwise_kernels` function to construct $K_{test}$. Then make predictions for the test set and evaluate the accuracy of our kernel logistic regression classifier. You should see a pretty substantial lift in accuracy to around $97\%$

In [17]:
# TODO
# Ktest = ...
Ktest = pairwise_kernels(Xts1, Xtr1, metric='rbf', gamma=gamma)


In [18]:
# TODO
# yhat = ... (vector containing predicted 0,1 labels)
yhat = np.dot(Ktest, alpha)
yhat = np.where(yhat > 0, 1, 0)
acc = np.mean(yhat == yts8)
print("Test accuracy = %f" % acc)
# acc = np.mean(yhat == yts8)
# print("Test accuracy = %f" % acc)

Test accuracy = 0.979800


## Kernel Support Vector Machine

The goal of this lab is to compare Kernel Logistic Regression to Kernel Support Vector machines. Following `demo_mnist_svm.ipynb` create and train an SVM classifier on `Xtr1` and `ytr8` using an RBF kernel with `gamma = .05` (the same value we used for logistic regression above). Use margin parameter `C = 10`.

In [19]:
# TODO

svc = svm.SVC(probability=False,  kernel="rbf", C=10, gamma=.05,verbose=10)
svc.fit(Xtr1, ytr8)

[LibSVM]

SVC(C=10, gamma=0.05, verbose=10)

Calculate and print the accuracy of the SVM classifier. You should obtain a similar result as for logistic regression: something close to $97\%$ accuracy.

In [20]:
# TODO
ysvm = svc.predict(Xts1)
# ysvm = ... (vector containing predicted 0,1 labels)
acc = np.mean(ysvm == yts8)
print("Test accuracy = %f" % acc)

Test accuracy = 0.980200


## The Showdown

Both SVM classifiers and kernel logisitic regression require tuning parameters to obtain the best possible result. In our setting we will stick with an RBF kernel (although this could be tuned). So we only consider tuning the kernel width parameter `gamma`, as well as the regularization parameter `lamb` for logistic regression, and the margin parameter `C` for SVM. We will choose parameters using for-loops and train-test cross validation.

Train a logistic regression classifier with **all combinations** of the parameters included below in vectors `gamma` and `lamb`. For each setting of parameters, compute and print:
* the test error obtained
* the total runtime of classification in seconds (including training time and prediction time)

For computing runtime you might want to use the `time()` function from the `time` library, which we already imported ealier.

In [21]:
gamma = [.1, .05,.02,.01,.005]
lamb = [0,1e-6,1e-4,1e-2]
results = []
for g in gamma:
  Ktest = pairwise_kernels(Xts1, Xtr1, metric='rbf', gamma=g)
  K = pairwise_kernels(Xtr1, metric='rbf', gamma=g)
  for l in lamb:
    start_time = time.time()
    alpha = log_fit(K, ytr8, lamb=l)
    yhat = np.dot(Ktest, alpha)
    yhat = np.where(yhat > 0, 1, 0)
    acc = np.mean(yhat == yts8)
    runtime = time.time() - start_time
    print(f"Gamma: {g}, Lambda: {l}")
    print(f"Test accuracy = %f" % acc)
    print(f"Runtime: {runtime} seconds")
    results.append((g, l, acc, runtime))

# results = sorted(results, key=lambda x: -x[2])
# print("Best parameters and results:")
# for g, l, acc, runtime in results:
#     print(f"Gamma: {g}, Lambda: {l}, Accuracy: {acc:.4f}, Runtime: {runtime:.4f}s")
# TODO
# ...

Gamma: 0.1, Lambda: 0
Test accuracy = 0.963200
Runtime: 1.8820006847381592 seconds
Gamma: 0.1, Lambda: 1e-06
Test accuracy = 0.965800
Runtime: 5.040554523468018 seconds
Gamma: 0.1, Lambda: 0.0001
Test accuracy = 0.965200
Runtime: 7.6075239181518555 seconds
Gamma: 0.1, Lambda: 0.01
Test accuracy = 0.962800
Runtime: 7.98199987411499 seconds
Gamma: 0.05, Lambda: 0
Test accuracy = 0.979800
Runtime: 3.67600154876709 seconds
Gamma: 0.05, Lambda: 1e-06
Test accuracy = 0.980000
Runtime: 17.370001316070557 seconds
Gamma: 0.05, Lambda: 0.0001
Test accuracy = 0.978000
Runtime: 44.00503492355347 seconds
Gamma: 0.05, Lambda: 0.01
Test accuracy = 0.974200
Runtime: 32.87805676460266 seconds


C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:15: RuntimeWarning: divide by zero encountered in log
  term2 = -np.sum(np.log(h))


Gamma: 0.02, Lambda: 0
Test accuracy = 0.981800
Runtime: 21.78351902961731 seconds
Gamma: 0.02, Lambda: 1e-06
Test accuracy = 0.985000
Runtime: 259.3252651691437 seconds
Gamma: 0.02, Lambda: 0.0001
Test accuracy = 0.984400
Runtime: 485.47258162498474 seconds
Gamma: 0.02, Lambda: 0.01
Test accuracy = 0.982800
Runtime: 295.37364625930786 seconds


C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:15: RuntimeWarning: divide by zero encountered in log
  term2 = -np.sum(np.log(h))


Gamma: 0.01, Lambda: 0
Test accuracy = 0.980200
Runtime: 62.73159217834473 seconds
Gamma: 0.01, Lambda: 1e-06
Test accuracy = 0.984600
Runtime: 985.5667355060577 seconds
Gamma: 0.01, Lambda: 0.0001
Test accuracy = 0.983000
Runtime: 994.916745185852 seconds
Gamma: 0.01, Lambda: 0.01
Test accuracy = 0.979600
Runtime: 806.6955661773682 seconds


C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
C:\Users\Dove0u0\AppData\Local\Temp\ipykernel_48064\2605267703.py:15: RuntimeWarning: divide by zero encountered in log
  term2 = -np.sum(np.log(h))


Gamma: 0.005, Lambda: 0
Test accuracy = 0.976200
Runtime: 204.81524968147278 seconds
Gamma: 0.005, Lambda: 1e-06
Test accuracy = 0.982400
Runtime: 999.1484320163727 seconds
Gamma: 0.005, Lambda: 0.0001
Test accuracy = 0.981800
Runtime: 990.7571201324463 seconds
Gamma: 0.005, Lambda: 0.01
Test accuracy = 0.976400
Runtime: 1012.5363953113556 seconds
Best parameters and results:
Gamma: 0.02, Lambda: 1e-06, Accuracy: 0.9850, Runtime: 259.3253s
Gamma: 0.01, Lambda: 1e-06, Accuracy: 0.9846, Runtime: 985.5667s
Gamma: 0.02, Lambda: 0.0001, Accuracy: 0.9844, Runtime: 485.4726s
Gamma: 0.01, Lambda: 0.0001, Accuracy: 0.9830, Runtime: 994.9167s
Gamma: 0.02, Lambda: 0.01, Accuracy: 0.9828, Runtime: 295.3736s
Gamma: 0.005, Lambda: 1e-06, Accuracy: 0.9824, Runtime: 999.1484s
Gamma: 0.02, Lambda: 0, Accuracy: 0.9818, Runtime: 21.7835s
Gamma: 0.005, Lambda: 0.0001, Accuracy: 0.9818, Runtime: 990.7571s
Gamma: 0.01, Lambda: 0, Accuracy: 0.9802, Runtime: 62.7316s
Gamma: 0.05, Lambda: 1e-06, Accuracy: 0.98

TODO: What was the best test error achieved, and what setting of parameters achieved this error? Was the kernel logistic regression classifier more sensitive to changes in `gamma` or `lamb`? Discuss in 1-3 short sentences below.

From the data I obtained here, Gamma: 0.02, Lambda: 1e-06 Test accuracy = 0.985000 is the best test error achieved. The Gamma influenced more to this classifier. 

Now let's do the same thing for the kernel Support Vector Classifier. Train an SVM classifier with **all combinations** of the parameters included below in vectors `gamma` and `C`. For each setting of parameters, compute:
* the test error obtained
* the total runtime of classification in seconds (including training time and prediction time)

In [24]:
gamma = [.1, .05,.02,.01,.005]
C = [.01,.1,1,10]
# TODO
# ...
results_svm = []

for g in gamma:
  for c in C:
    start_time = time.time()
    svc = svm.SVC(probability=False,  kernel="rbf", C=c, gamma=g,verbose=10)
    svc.fit(Xtr1, ytr8)
    ysvm = svc.predict(Xts1)
    acc = np.mean(ysvm == yts8)
    runtime = time.time() - start_time
    print(f"Gamma: {g}, C: {c}")
    print(f"Test accuracy = %f" % acc)
    print(f"Runtime: {runtime} seconds")
    results_svm.append((g, c, acc, runtime))

# results_svm  = sorted(results, key=lambda x: -x[2])
# print("Best parameters and results:")
# for g, c, acc, runtime in results:
#   print(f"Gamma: {g}, C: {c}, Accuracy: {acc:.4f}, Runtime: {runtime:.4f}s")



[LibSVM]Gamma: 0.1, C: 0.01
Test accuracy = 0.902200
Runtime: 36.8384850025177 seconds
[LibSVM]Gamma: 0.1, C: 0.1
Test accuracy = 0.902200
Runtime: 54.90621304512024 seconds
[LibSVM]Gamma: 0.1, C: 1
Test accuracy = 0.928200
Runtime: 69.64148902893066 seconds
[LibSVM]Gamma: 0.1, C: 10
Test accuracy = 0.934200
Runtime: 75.90764904022217 seconds
[LibSVM]Gamma: 0.05, C: 0.01
Test accuracy = 0.902200
Runtime: 11.418000221252441 seconds
[LibSVM]Gamma: 0.05, C: 0.1
Test accuracy = 0.922800
Runtime: 12.986051321029663 seconds
[LibSVM]Gamma: 0.05, C: 1
Test accuracy = 0.977400
Runtime: 15.614859580993652 seconds
[LibSVM]Gamma: 0.05, C: 10
Test accuracy = 0.980200
Runtime: 16.630459308624268 seconds
[LibSVM]Gamma: 0.02, C: 0.01
Test accuracy = 0.902200
Runtime: 7.599788188934326 seconds
[LibSVM]Gamma: 0.02, C: 0.1
Test accuracy = 0.951200
Runtime: 6.661513328552246 seconds
[LibSVM]Gamma: 0.02, C: 1
Test accuracy = 0.982000
Runtime: 5.251333951950073 seconds
[LibSVM]Gamma: 0.02, C: 10
Test accura

TODO: What was the best test error achieved, and what setting of parameters achieved this error? Which performed better in terms of accuracy, the SVM or logisitic regression classifier? How about in terms of runtime? **Add some discussion here.**

The best error achieved is with LibSVM]Gamma: 0.01, C: 10 Test accuracy = 0.987000 which is higher than log classifier, but the log one has a higher accuracy average but it takes more time than SVM.

**NOTE:** For `sklearns`'s built in classifiers, including svm.SVC, there is a function called `GridSearchCV` which can automatically perform hyperparamater tuning for you. The main advantage of the method (as opposed to writing for-loops) is that it supports parallelization, so it can fit with different parameters at the same time.